# DQ 检查 — ODS / DWD 层（Rules 1-12）

Rules 1-10: ODS 完整性/范围/时序
Rules 11-12: DWD 行数守恒 + 参照完整性

任何 FAIL 将抛出异常，阻断 DWS 下游任务。

## 1. SparkSession

In [ ]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterDWD")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE          = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y         = f"{BASE}/data/warehouse/ods/yellow"
ODS_G         = f"{BASE}/data/warehouse/ods/green"
ODS_F         = f"{BASE}/data/warehouse/ods/fhv_hv"
DWD_FACT      = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE      = f"{BASE}/data/warehouse/dwd/dim_zone"
DWD_CLEANING  = f"{BASE}/data/warehouse/dq/dwd_cleaning"   # records rejected by DWD cleaning
QUARANTINE    = f"{BASE}/data/warehouse/dq/quarantine"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


## 2. DQ 框架

In [2]:
results = []

def check(rule_id, layer, rule_type, description, passed, bad_count=0, total=0, threshold=None):
    status = "PASS" if passed else "FAIL"
    results.append({
        "rule_id": rule_id, "layer": layer, "rule_type": rule_type,
        "description": description, "status": status,
        "bad_count": bad_count, "total": total, "threshold": threshold
    })
    mark = "✅" if passed else "❌"
    print(f"{mark} Rule {rule_id:02d} [{layer}] {description}  bad={bad_count:,}")


## 3. Rules 1-10：ODS 层检查（Yellow + Green）
FHV HV 字段名不同（base_passenger_fare / trip_miles / PUlocationID），单独在第 5 节检查。

In [3]:
ods_y = spark.read.parquet(ODS_Y).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
ods_g = spark.read.parquet(ODS_G).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
ods   = ods_y.unionByName(ods_g, allowMissingColumns=True)
ods_total = ods.count()
print(f"ODS (yellow+green) total: {ods_total:,}")

# FHV HV 单独读取，字段名不同，后续独立检查
ods_f = spark.read.parquet(ODS_F).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
ods_f_total = ods_f.count()
print(f"ODS (fhv_hv) total:      {ods_f_total:,}")

# Rules 1-4: 关键字段严格不允许 null
for rule_id, field in enumerate(
    ["fare_amount", "pickup_datetime", "PULocationID", "DOLocationID"],
    start=1
):
    bad = ods.filter(col(field).isNull()).count()
    check(rule_id, "ODS", "completeness", f"{field} not null",
          passed=(bad == 0), bad_count=bad, total=ods_total)

# Rule 5: passenger_count null 率告警（< 15%）
# null 记录车费/时间/区域均完整，属真实行程 → 打标，不隔离
bad5 = ods.filter(col("passenger_count").isNull()).count()
check(5, "ODS", "completeness_warn",
      "passenger_count null rate < 15% (tag-only, not quarantined)",
      passed=(bad5 / ods_total < 0.15), bad_count=bad5, total=ods_total, threshold=0.15)

# Rule 6: fare_amount > 0
bad6 = ods.filter(col("fare_amount") <= 0).count()
check(6, "ODS", "range", "fare_amount > 0", passed=(bad6 / ods_total < 0.05),
      bad_count=bad6, total=ods_total, threshold=0.05)

# Rule 7: trip_distance >= 0
bad7 = ods.filter(col("trip_distance") < 0).count()
check(7, "ODS", "range", "trip_distance >= 0", passed=(bad7 == 0),
      bad_count=bad7, total=ods_total)

# Rule 8: passenger_count in [1,6]（null 单独由 Rule 5 处理，此处只检查非 null 越界）
bad8 = ods.filter(
    col("passenger_count").isNotNull() & ~col("passenger_count").between(1, 6)
).count()
check(8, "ODS", "range", "passenger_count in [1,6]", passed=(bad8 / ods_total < 0.15),
      bad_count=bad8, total=ods_total, threshold=0.15)

# Rule 9: dropoff > pickup
bad9 = ods.filter(col("dropoff_datetime") <= col("pickup_datetime")).count()
check(9, "ODS", "temporal", "dropoff > pickup", passed=(bad9 / ods_total < 0.01),
      bad_count=bad9, total=ods_total)

# Rule 10: 时间在 2024 Q1
bad10 = ods.filter(
    (col("pickup_datetime") < "2024-01-01") | (col("pickup_datetime") >= "2024-04-01")
).count()
check(10, "ODS", "temporal", "pickup in 2024-Q1", passed=(bad10 / ods_total < 0.01),
      bad_count=bad10, total=ods_total)


ODS (yellow+green) total: 130,849


ODS (fhv_hv) total:      740,231


✅ Rule 01 [ODS] fare_amount not null  bad=0


✅ Rule 02 [ODS] pickup_datetime not null  bad=0


✅ Rule 03 [ODS] PULocationID not null  bad=0


...

✅ Rule 06 [ODS] fare_amount > 0  bad=2,246


✅ Rule 07 [ODS] trip_distance >= 0  bad=0


✅ Rule 08 [ODS] passenger_count in [1,6]  bad=1,516


✅ Rule 09 [ODS] dropoff > pickup  bad=40


✅ Rule 10 [ODS] pickup in 2024-Q1  bad=0


In [4]:
# ── Rules 13-16：ODS 层检查（FHV HV 专属字段）─────────────────────────────
# 网约车特有：trip_time（行程秒数）、driver_pay（司机收入）、request_datetime（叫车时间）
# 补充这些规则，使两类数据源接受对等强度的检查，避免 0.01% 隔离率被误解为"更干净"
fhv_cols = set(ods_f.columns)
print(f"FHV HV columns: {sorted(fhv_cols)}\n")

if "trip_time" in fhv_cols:
    bad13 = ods_f.filter(col("trip_time") <= 0).count()
    check(13, "ODS_FHV", "range", "trip_time > 0",
          passed=(bad13 / ods_f_total < 0.01), bad_count=bad13, total=ods_f_total)
else:
    print("⚠️  trip_time not found — Rule 13 skipped")

if "trip_miles" in fhv_cols:
    bad14 = ods_f.filter(col("trip_miles") <= 0).count()
    check(14, "ODS_FHV", "range", "trip_miles > 0",
          passed=(bad14 / ods_f_total < 0.01), bad_count=bad14, total=ods_f_total)
else:
    print("⚠️  trip_miles not found — Rule 14 skipped")

if "driver_pay" in fhv_cols:
    bad15 = ods_f.filter(col("driver_pay") < 0).count()
    check(15, "ODS_FHV", "range", "driver_pay >= 0",
          passed=(bad15 / ods_f_total < 0.01), bad_count=bad15, total=ods_f_total)
else:
    print("⚠️  driver_pay not found — Rule 15 skipped")

if "request_datetime" in fhv_cols:
    bad16 = ods_f.filter(
        col("request_datetime").isNotNull() &
        (col("request_datetime") > col("pickup_datetime"))
    ).count()
    check(16, "ODS_FHV", "temporal", "request_datetime <= pickup_datetime",
          passed=(bad16 / ods_f_total < 0.01), bad_count=bad16, total=ods_f_total)
else:
    print("⚠️  request_datetime not found — Rule 16 skipped")


FHV HV columns: ['DOLocationID', 'PULocationID', 'access_a_ride_flag', 'airport_fee', 'base_passenger_fare', 'bcf', 'congestion_surcharge', 'dispatching_base_num', 'driver_pay', 'dropoff_datetime', 'hvfhs_license_num', 'ingestion_ts', 'on_scene_datetime', 'originating_base_num', 'pickup_datetime', 'pickup_day', 'pickup_month', 'pickup_year', 'request_datetime', 'sales_tax', 'shared_match_flag', 'shared_request_flag', 'source_type', 'tips', 'tolls', 'trip_miles', 'trip_time', 'wav_match_flag', 'wav_request_flag']

✅ Rule 13 [ODS_FHV] trip_time > 0  bad=0


✅ Rule 14 [ODS_FHV] trip_miles > 0  bad=100


✅ Rule 15 [ODS_FHV] driver_pay >= 0  bad=6


✅ Rule 16 [ODS_FHV] request_datetime <= pickup_datetime  bad=7,085


## 4. Rules 11-12：DWD 层检查

In [ ]:
fact       = spark.read.parquet(DWD_FACT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
fact_count = fact.count()
dim_zone   = spark.read.parquet(DWD_ZONE)
print(f"fact_trips: {fact_count:,}")

# Rule 11: True conservation — fact + dwd_cleaning == ods_all_total (no silent drops)
# dwd_cleaning holds records rejected during DWD cleaning (written by 03_dwd_build)
# If this check fails, records were silently dropped — a bug, not a quality filter
try:
    dwd_cleaning = spark.read.parquet(DWD_CLEANING).filter(
        (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
    rejected_count = dwd_cleaning.count()
except Exception:
    rejected_count = 0
    print("⚠️  DWD_CLEANING not found — run 03_dwd_build first to populate it")

ods_all_total  = ods_total + ods_f_total
accounted_rows = fact_count + rejected_count
gap            = ods_all_total - accounted_rows

conservation_ok = (gap == 0)
check(11, "DWD", "row_conservation",
      f"fact ({fact_count:,}) + dwd_cleaning ({rejected_count:,}) == ods_all ({ods_all_total:,})",
      passed=conservation_ok,
      bad_count=abs(gap), total=ods_all_total)

if not conservation_ok:
    print(f"  ⚠️  {abs(gap):,} records {'silently dropped' if gap > 0 else 'extra'} — "
          "check 03_dwd_build for untracked filter paths")

# Rule 12: 参照完整性 PULocationID ∈ dim_zone
valid_zones = set(r.location_id for r in dim_zone.collect())
bad12 = fact.filter(~col("PULocationID").isin(list(valid_zones))).count()
check(12, "DWD", "referential", "PULocationID in dim_zone",
      passed=(bad12 == 0), bad_count=bad12, total=fact_count)


## 5. 违规记录写入隔离表

In [ ]:
fhv_cols = set(ods_f.columns)

# Yellow + Green 隔离记录
# passenger_count IS NULL 改为打标（passenger_count_missing），不再隔离
quarantine_yg = (
    ods
    .withColumn("passenger_count_missing", col("passenger_count").isNull())
    .withColumn("dq_reason",
        F.when(col("fare_amount").isNull(),          "null_fare_amount")
        .when(col("pickup_datetime").isNull(),        "null_pickup_datetime")
        .when(col("PULocationID").isNull(),           "null_PULocationID")
        .when(col("DOLocationID").isNull(),           "null_DOLocationID")
        .when(col("fare_amount") <= 0,                "non_positive_fare")
        .when(col("trip_distance") < 0,               "negative_distance")
        .when(col("passenger_count").isNotNull() &
              ~col("passenger_count").between(1, 6),  "invalid_passenger_count")
        .when(col("dropoff_datetime") <= col("pickup_datetime"), "invalid_time_order")
        .when(
            (col("pickup_datetime") < "2024-01-01") |
            (col("pickup_datetime") >= "2024-04-01"), "out_of_range_datetime")
    )
    .filter(col("dq_reason").isNotNull())
    .withColumn("check_ts", F.current_timestamp())
)

# FHV HV 隔离记录（含网约车专属规则）
fhv_reason = (
    F.when(col("fare_amount").isNull(),    "null_fare_amount")
    .when(col("pickup_datetime").isNull(), "null_pickup_datetime")
    .when(col("PULocationID").isNull(),    "null_PULocationID")
    .when(col("DOLocationID").isNull(),    "null_DOLocationID")
    .when(col("fare_amount") <= 0,         "non_positive_fare")
    .when(col("trip_distance") <= 0,       "non_positive_trip_distance")
    .when(col("dropoff_datetime") <= col("pickup_datetime"), "invalid_time_order")
)
if "trip_time" in fhv_cols:
    fhv_reason = fhv_reason.when(col("trip_time") <= 0, "non_positive_trip_time")
if "driver_pay" in fhv_cols:
    fhv_reason = fhv_reason.when(col("driver_pay") < 0, "negative_driver_pay")
if "request_datetime" in fhv_cols:
    fhv_reason = fhv_reason.when(
        col("request_datetime").isNotNull() &
        (col("request_datetime") > col("pickup_datetime")),
        "request_after_pickup"
    )

quarantine_fhv = (
    ods_f
    .withColumnRenamed("base_passenger_fare", "fare_amount")
    .withColumnRenamed("trip_miles",          "trip_distance")
    .withColumnRenamed("PUlocationID",        "PULocationID")
    .withColumnRenamed("DOlocationID",        "DOLocationID")
    .withColumn("dq_reason", fhv_reason)
    .filter(col("dq_reason").isNotNull())
    .withColumn("check_ts", F.current_timestamp())
)

q_yg_count  = quarantine_yg.count()
q_fhv_count = quarantine_fhv.count()
total_q     = q_yg_count + q_fhv_count
ods_all     = ods_total + ods_f_total
q_rate      = total_q / ods_all

print(f"Yellow+Green: {q_yg_count:,} / {ods_total:,} = {q_yg_count/ods_total:.2%}")
print(f"FHV HV:       {q_fhv_count:,} / {ods_f_total:,} = {q_fhv_count/ods_f_total:.2%}")
print(f"Overall:      {total_q:,} / {ods_all:,} = {q_rate:.2%}")

print("\nYellow+Green breakdown:")
quarantine_yg.groupBy("dq_reason").count().orderBy(F.desc("count")).show()
print("FHV HV breakdown:")
quarantine_fhv.groupBy("dq_reason").count().orderBy(F.desc("count")).show()

# 动态分区覆盖：按 source_type + 日期分区，保留历史每日快照
(quarantine_yg.write.mode("overwrite")
    .partitionBy("source_type", "pickup_year", "pickup_month", "pickup_day")
    .parquet(QUARANTINE))
(quarantine_fhv.write.mode("overwrite")
    .partitionBy("source_type", "pickup_year", "pickup_month", "pickup_day")
    .parquet(QUARANTINE))
print(f"Quarantine written to: {QUARANTINE}")


## 6. DQ 报告 + 阻断判断

In [7]:
fail_rules = [r for r in results if r['status'] == 'FAIL']
pass_rules = [r for r in results if r['status'] == 'PASS']

print(f"\n{'='*50}")
print(f"DQ Report (ODS+DWD)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(fail_rules)} FAIL")
print(f"Quarantine rate: {q_rate:.2%}")
print(f"={'='*49}")

spark.stop()

if fail_rules:
    print("FAIL rules (blocking):")
    for r in fail_rules:
        print(f"  ❌ Rule {r['rule_id']:02d}: {r['description']}  bad={r['bad_count']:,}")
    raise ValueError(
        f"DQ FAILED: {len(fail_rules)} blocking rule(s). "
        "Pipeline halted — fix upstream data before proceeding to DWS."
    )
else:
    print("All ODS/DWD rules passed — proceeding to DWS.")

print("DQ after-DWD check complete.")



DQ Report (ODS+DWD)  2026-10-19 16:25
Rules: 16 total | 16 PASS | 0 FAIL
Quarantine rate: 1.27%


All ODS/DWD rules passed — proceeding to DWS.
DQ after-DWD check complete.
